In [1]:
import os
import random
from data.preprocessing.pipeline import Pipeline

In [2]:
pipeline = Pipeline(feature_type="rdkit+dft", use_pca_rdkit=False, use_pca_dft=False, remove_zero_var=True)
train_df, test_df = pipeline.get_processed_data()

Extracting raw features:   0%|          | 0/2064 [00:00<?, ?it/s][19:44:36] DEPRECATION WARNING: please use MorganGenerator
[19:44:36] DEPRECATION WARNING: please use MorganGenerator
[19:44:36] DEPRECATION WARNING: please use MorganGenerator
[19:44:36] DEPRECATION WARNING: please use MorganGenerator
[19:44:36] DEPRECATION WARNING: please use MorganGenerator
[19:44:36] DEPRECATION WARNING: please use MorganGenerator
[19:44:36] DEPRECATION WARNING: please use MorganGenerator
Extracting raw features:   0%|          | 7/2064 [00:00<00:30, 66.80it/s][19:44:36] DEPRECATION WARNING: please use MorganGenerator
[19:44:36] DEPRECATION WARNING: please use MorganGenerator
[19:44:36] DEPRECATION WARNING: please use MorganGenerator
[19:44:36] DEPRECATION WARNING: please use MorganGenerator
[19:44:36] DEPRECATION WARNING: please use MorganGenerator
[19:44:37] DEPRECATION WARNING: please use MorganGenerator
[19:44:37] DEPRECATION WARNING: please use MorganGenerator
[19:44:37] DEPRECATION WARNING: plea

In [9]:
train_df

,Pubchem_ID,taste_cluster,SMILES,G,HOMO,LUMO,Gap,Dipole,E,CavityVolume,...,feat_169,feat_170,feat_171,feat_172,feat_173,feat_174,feat_175,feat_176,feat_177,feat_178
0,95609,0,CCC(C(=O)CC)O,-27.094628,-10.8128,-6.9887,0.140533,3.869,-27.369769,122.600,...,-0.125491,-0.508118,0.998663,-0.201499,-0.401680,-0.353943,0.299934,1.003197,-1.023515,-0.967617
1,104224,0,CCCCCCCCCCC(OC)OC,-50.265999,-10.9348,0.2265,0.410171,2.457,-50.795266,244.360,...,-0.125491,2.126945,-1.062374,-0.450322,2.637220,2.683741,-0.414344,-1.056548,1.174718,1.227360
2,5284499,6,CCCCCCCC/C=C\CCCCCCCCO,-60.970606,-10.3723,-5.2295,0.188996,2.134,-61.746753,313.496,...,-0.125491,4.009134,-2.014522,0.696918,0.339261,0.192028,-0.577738,-2.019484,2.422887,2.540599
3,10886,4,CCCCCC(=O)OCCCCC,-42.945972,-11.4100,-6.4368,0.182764,3.073,-43.418595,207.360,...,-0.125491,0.997633,-0.411275,-1.419509,-0.169230,0.121821,-0.102733,-0.407936,0.506727,0.696595
4,12978217,3,C/C=C/C(=O)[C@@H]1[C@H](C=CCC1(C)C)C,-42.076808,-10.3676,-7.6082,0.101403,4.148,-42.644300,204.552,...,-0.125491,-0.508118,-0.333965,0.706504,-0.662601,-0.794765,0.441069,-0.339854,0.456032,0.251927
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2059,229385,1,CCCCCCCCCCOC(=O)CC,-49.270330,-11.0605,-6.3783,0.172070,3.180,-49.812813,238.664,...,-0.125491,2.126945,-0.973811,-0.706692,-0.144591,0.001345,-0.048605,-0.970163,1.071884,1.024778
2060,62387,2,CC1(C2CCC1([C@H](C2)OC=O)C)C,-40.916082,-11.0892,-6.7654,0.158897,3.773,-41.362330,181.808,...,-0.125491,-0.508118,-0.230722,-0.765226,-0.307630,-0.147055,0.251371,-0.227134,0.045415,-0.216544
2061,8878,7,CCCCCCC(=O)OCC=C,-38.719417,-11.2611,-6.4675,0.176160,2.859,-39.175686,184.464,...,-0.125491,0.621195,-0.035334,-1.115822,-0.182161,0.047429,-0.210987,-0.034867,0.093366,0.212424
2062,61262,1,CC(=O)OCC1=CCC2CC1C2(C)C,-43.019564,-10.3913,-6.3856,0.147207,2.705,-43.514330,195.840,...,-0.125491,-0.508118,-0.417821,0.658167,-0.147666,-0.278746,-0.288890,-0.416353,0.298747,0.144052


In [17]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

X_desc = train_df.drop(columns=['Pubchem_ID', 'taste_cluster', 'SMILES'])
y = train_df['taste_cluster']

X_train, X_val, y_train, y_val, train_smiles, val_smiles = train_test_split(
    X_desc, y, train_df['SMILES'], test_size=0.2, random_state=42, stratify=y
)

In [18]:
from rdkit import Chem
import torch
from torch_geometric.data import Data, DataLoader
from torch_geometric.nn import GCNConv, global_mean_pool
import torch.nn.functional as F
import torch.nn as nn

# Функция SMILES → граф
def smiles_to_graph(smiles):
    mol = Chem.MolFromSmiles(smiles)
    x = torch.tensor([[atom.GetAtomicNum()] for atom in mol.GetAtoms()], dtype=torch.float)
    edge_index = []
    for bond in mol.GetBonds():
        edge_index.append([bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()])
        edge_index.append([bond.GetEndAtomIdx(), bond.GetBeginAtomIdx()])
    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
    return Data(x=x, edge_index=edge_index)

# Преобразуем SMILES для train/val
train_graphs = [smiles_to_graph(s) for s in train_smiles]
val_graphs = [smiles_to_graph(s) for s in val_smiles]

train_loader = DataLoader(train_graphs, batch_size=32, shuffle=True)
val_loader = DataLoader(val_graphs, batch_size=32)


C:\Users\edtop\AppData\Local\Temp\ipykernel_25876\942372243.py:23: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  train_loader = DataLoader(train_graphs, batch_size=32, shuffle=True)
C:\Users\edtop\AppData\Local\Temp\ipykernel_25876\942372243.py:24: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  val_loader = DataLoader(val_graphs, batch_size=32)


In [19]:
class GNN(nn.Module):
    def __init__(self, hidden_dim=64, num_classes=len(y.unique())):
        super().__init__()
        self.conv1 = GCNConv(1, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.lin = nn.Linear(hidden_dim, num_classes)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        x = global_mean_pool(x, batch)
        out = self.lin(x)
        return out

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
gnn = GNN().to(device)
optimizer = torch.optim.Adam(gnn.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

In [20]:
gnn.train()
y_train_tensor = torch.tensor(y_train.values, dtype=torch.long).to(device)

for epoch in range(5):  # для примера, можно больше
    total_loss = 0
    idx = 0
    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        out = gnn(batch)
        batch_size = batch.num_graphs
        loss = criterion(out, y_train_tensor[idx:idx+batch_size])
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        idx += batch_size
    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}")


Epoch 1, Loss: 1.9342
Epoch 2, Loss: 1.9071
Epoch 3, Loss: 1.9071
Epoch 4, Loss: 1.9067
Epoch 5, Loss: 1.9071


In [21]:
def get_gnn_embeddings(model, loader):
    model.eval()
    embeddings = []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            x = F.relu(model.conv1(batch.x, batch.edge_index))
            x = F.relu(model.conv2(x, batch.edge_index))
            x = global_mean_pool(x, batch.batch)
            embeddings.append(x.cpu().numpy())
    return np.vstack(embeddings)

gnn_train_emb = get_gnn_embeddings(gnn, train_loader)
gnn_val_emb = get_gnn_embeddings(gnn, val_loader)

In [ ]:
from sklearn.metrics import f1_score
from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
import numpy as np

# XGBoost
xgb = XGBClassifier(n_estimators=200, max_depth=6, random_state=42, eval_metric='mlogloss')
xgb.fit(X_train, y_train)

# SVM
svm = SVC(probability=True, kernel='rbf', C=10)
svm.fit(X_train, y_train)

# Получаем вероятности на валидации
xgb_val_preds = xgb.predict_proba(X_val)
svm_val_preds = svm.predict_proba(X_val)

# Стэкинг + GNN эмбеддинги
stacked_val = np.hstack([xgb_val_preds, svm_val_preds, gnn_val_emb])
meta_clf = LogisticRegression(max_iter=1000)
meta_clf.fit(stacked_val, y_val)

# Финальные предсказания
final_val_preds = meta_clf.predict(stacked_val)

# F1 Macro
f1_macro = f1_score(y_val, final_val_preds, average='macro')
print(f"F1 Macro на валидации: {f1_macro:.4f}")

F1 Macro на валидации: 0.3971
